In [ ]:
from si4dnn.CPU import parse_model
from si4dnn import InferenceModel
import torch
import numpy as np
import time

In [ ]:
model = torch.nn.Sequential(
        torch.nn.Linear(5, 100),
        torch.nn.ReLU(),
        torch.nn.Linear(100, 100),
        torch.nn.ReLU(),
        # torch.nn.Linear(500, 500),
        # torch.nn.ReLU(),
        # torch.nn.Linear(500, 500),
        # torch.nn.ReLU(),
        # torch.nn.Linear(500, 500),
        # torch.nn.ReLU(),
        # torch.nn.Linear(500, 500),
        # torch.nn.ReLU(),
        torch.nn.Linear(100, 2)
    ) # Can be modified freely

In [ ]:
# Training process
n_train = 1000
p = 5

# Generate synthetic training data
X_train = torch.randn(n_train, p)
y_train = torch.randn(n_train, 2)  # Target with 2 outputs to match model output

# Setup optimizer and loss function
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = torch.nn.MSELoss()

# Training loop
n_epochs = 10
batch_size = 64

model.train()
for epoch in range(n_epochs):
    # Shuffle data
    perm = torch.randperm(n_train)
    X_train = X_train[perm]
    y_train = y_train[perm]
    
    X_train = X_train
    y_train = y_train
    
    epoch_loss = 0.0
    n_batches = 0
    
    for i in range(0, n_train, batch_size):
        X_batch = X_train[i:i+batch_size]
        y_batch = y_train[i:i+batch_size]
        
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
        n_batches += 1
    
    if (epoch + 1) % 20 == 0:
        print(f'Epoch [{epoch+1}/{n_epochs}], Loss: {epoch_loss/n_batches:.4f}')

model.eval()
print('Training completed!')

In [ ]:
n = 50
p = 5


inference_model = InferenceModel(model, device='cuda') # Can be "cuda" also

# X = a + bz
a = np.random.randn(n, p).astype(np.float32)
b = np.random.randn(n, p).astype(np.float32)
z = np.random.randn(1).astype(np.float32)
X = a + b * z

In [ ]:
# Find interval for the forward process and X_hat = model(X) = a_hat + b_hat * z 
X_hat = model(torch.from_numpy(X))
start = time.time()
a_hat, b_hat, model_itv, = inference_model.forward(a, b, z)
print("Time for forward process:", time.time() - start)
print(model_itv)

In [ ]:
itv = [-20, 20]
start = time.time()
z = -20
while z <= 20:
    X = a + b * z
    X_hat = model(torch.from_numpy(X))
    a_hat, b_hat, model_itv = inference_model.forward(a, b, np.array([z], dtype=np.float32))
    print(f"z: {z}, model_itv: {model_itv}")
    z = model_itv[1] + 1e-5
print("Time for interval covering:", time.time() - start)
print('Length of intervals:', len(model_itv))

In [ ]:
class Interval:
    def __init__(self, lower, upper, w, b):
        self.lower = lower
        self.upper = upper
        self.w = w 
        self.b = b

In [ ]:
def get_model_intervals(model, intervals):

    layers = parse_model(model)
    
    # Each interval: (left, right, a_curr, b_curr)
    # a_curr, b_curr represent the current effective linear form at this point
    # in the network: output = a_curr + b_curr * z
    # intervals = [(L, R, a.copy(), b.copy())]
    
    for layer_type, params in layers:
        if layer_type == 'Linear':
            W, bias = params  # W: (input_dim, output_dim)
            # Apply linear transformation to each interval's stored (a_curr, b_curr)
            new_intervals = []
            for (left, right, a_curr, b_curr) in intervals:
                # Linear layer: output = input @ W + bias
                a_new = a_curr @ W
                if bias is not None:
                    a_new = a_new + bias
                b_new = b_curr @ W
                new_intervals.append((left, right, a_new, b_new))
            intervals = new_intervals
            
        elif layer_type == 'ReLU':
            new_intervals = []
            

            num_intervals = len(intervals)
            if num_intervals == 0:
                continue

            # Stack all data
            lefts = np.array([left for left, _, _, _ in intervals])
            rights = np.array([right for _, right, _, _ in intervals])
            A_curr = np.array([a_curr for _, _, a_curr, _ in intervals])
            B_curr = np.array([b_curr for _, _, _, b_curr in intervals])

            # Compute all z_stars
            with np.errstate(divide='ignore', invalid='ignore'):
                Z_star_matrix = np.where(np.abs(B_curr) >= 1e-12, -A_curr / B_curr, np.inf)

            # Validity mask
            in_range_matrix = (Z_star_matrix > lefts[:, np.newaxis, np.newaxis]) & (Z_star_matrix < rights[:, np.newaxis, np.newaxis])

            # Collect all valid splits per interval
            new_intervals = []
            for idx in range(num_intervals):
                left = lefts[idx]
                right = rights[idx]
                a_curr = A_curr[idx]
                b_curr = B_curr[idx]
                
                # Get interior splits
                interior_splits = Z_star_matrix[idx][in_range_matrix[idx]]
                
                if interior_splits.size > 0:
                    splits = np.unique(np.concatenate([[left, right], interior_splits]))
                else:
                    splits = np.array([left, right])
                
                # Vectorized sub-interval processing
                num_splits = len(splits) - 1
                sub_lefts = splits[:-1]
                sub_rights = splits[1:]
                z_mids = (sub_lefts + sub_rights) * 0.5
                
                # Broadcast: z_mids shape (num_splits,), a_curr/b_curr shape (neurons,)
                # Result: pre_activations shape (num_splits, neurons)

                pre_activations = a_curr + b_curr * z_mids[:, np.newaxis, np.newaxis]
                active_masks = (pre_activations > 0).astype(np.float64)
                
                # Apply masks
                a_news = a_curr * active_masks
                b_news = b_curr * active_masks
                
                # Append all sub-intervals
                for i in range(num_splits):
                    new_intervals.append((sub_lefts[i], sub_rights[i], a_news[i], b_news[i]))

            intervals = new_intervals
            
        elif layer_type == 'LeakyReLU':
            alpha = params
            new_intervals = []
            
            for (left, right, a_curr, b_curr) in intervals:
                # Compute all z_star values at once
                with np.errstate(divide='ignore', invalid='ignore'):
                    z_stars = np.where(np.abs(b_curr) >= 1e-12, -a_curr / b_curr, np.inf)
                
                # Filter to those strictly inside the interval
                in_range = (z_stars > left) & (z_stars < right)
                interior_splits = z_stars[in_range]
                
                # Use np.unique for sorting and deduplication
                if interior_splits.size > 0:
                    splits = np.unique(np.concatenate([[left, right], interior_splits]))
                else:
                    splits = np.array([left, right])
                
                for i in range(len(splits) - 1):
                    sub_left = splits[i]
                    sub_right = splits[i + 1]
                    z_mid = (sub_left + sub_right) * 0.5
                    
                    pre_activation = a_curr + b_curr * z_mid
                    active_mask = np.where(pre_activation > 0, 1.0, 0.0)
                    inactive_mask = 1.0 - active_mask
                    
                    # LeakyReLU: x if x > 0 else alpha * x
                    a_new = a_curr * active_mask + alpha * a_curr * inactive_mask
                    b_new = b_curr * active_mask + alpha * b_curr * inactive_mask
                    
                    new_intervals.append((sub_left, sub_right, a_new, b_new))
            
            intervals = new_intervals
    intervals = sorted(intervals, key=lambda x: x[0])
    
    
    return intervals


In [ ]:
intervals = [(-20, 20, a, b)]
intervals = get_model_intervals(model, intervals)